# Netflix Data Science Tasks (Tasks 1–4)
**Auspify Data Science Internship**

| Task | Level | Topic |
|---|---|---|
| 1 | Easy | Data Cleaning & Preprocessing |
| 2 | Easy | Exploratory Data Analysis (EDA) |
| 3 | Medium | Recommendation System Analysis |
| 4 | Medium | Trend Prediction Analysis |

Keep `Dataset.csv` in the same folder as this notebook, then run the cells from top to bottom.

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("outputs", exist_ok=True)   # charts and the clean dataset are saved here

---
# Task 1 (Easy): Data Cleaning & Preprocessing
## Step 1: Import the dataset using Pandas

In [ ]:
df = pd.read_csv("Dataset.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## Step 2: Identify missing and duplicate records

In [ ]:
print("Null values per column:")
print(df.isna().sum())

# This dataset has no real NaNs, but it uses the text "Not Given" as a placeholder
print("\n'Not Given' placeholders per column:")
print((df == "Not Given").sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate show_id:", df["show_id"].duplicated().sum())

# Same title/details but a different show_id (hidden duplicates)
dups_ignoring_id = df.drop(columns="show_id").duplicated().sum()
print("Duplicates when show_id is ignored:", dups_ignoring_id)
df[df.drop(columns="show_id").duplicated(keep=False)].sort_values("title")

## Step 3: Handle null values and inconsistent formats

In [ ]:
clean = df.copy()

# 3a. Remove leading/trailing spaces in all text columns
text_cols = clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    clean[c] = clean[c].str.strip()

# 3b. Treat "Not Given" as missing, then label it "Unknown"
clean = clean.replace("Not Given", np.nan)
clean["director"] = clean["director"].fillna("Unknown")
clean["country"] = clean["country"].fillna("Unknown")

# 3c. Drop the hidden duplicates (same content, different show_id)
before = len(clean)
clean = clean.drop_duplicates(subset=clean.columns.drop("show_id"), keep="first")
print(f"Removed {before - len(clean)} duplicate rows")

# 3d. Fix the date format: text like 9/25/2021 -> real datetime
clean["date_added"] = pd.to_datetime(clean["date_added"], format="%m/%d/%Y", errors="coerce")
print("Dates that failed to parse:", clean["date_added"].isna().sum())

# 3e. Check categorical columns for odd values
print("\nTypes:", clean["type"].unique())
print("Ratings:", sorted(clean["rating"].unique()))

## Step 4: Transform categorical and date-related columns

In [ ]:
# ---- Date features
clean["year_added"]  = clean["date_added"].dt.year
clean["month_added"] = clean["date_added"].dt.month
clean["month_name"]  = clean["date_added"].dt.month_name()
clean["years_to_add"] = clean["year_added"] - clean["release_year"]   # lag between release and Netflix

# ---- Duration: "90 min" / "2 Seasons" -> number + unit
dur = clean["duration"].str.extract(r"(\d+)\s*(\w+)")
clean["duration_value"] = dur[0].astype(int)
clean["duration_unit"]  = dur[1].replace({"Seasons": "Season"})      # min or Season

# ---- Multi-value columns: keep the first value as the main one
clean["primary_country"] = clean["country"].str.split(",").str[0].str.strip()
clean["genres"]          = clean["listed_in"].str.split(",").apply(lambda g: [x.strip() for x in g])
clean["primary_genre"]   = clean["genres"].str[0]
clean["num_genres"]      = clean["genres"].str.len()

# ---- Group the many rating labels into audience categories
rating_map = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Family", "PG": "Family",
    "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
    "NR": "Unrated", "UR": "Unrated",
}
clean["rating_group"] = clean["rating"].map(rating_map)

# ---- Encode the type column (0 = Movie, 1 = TV Show)
clean["type_encoded"] = clean["type"].map({"Movie": 0, "TV Show": 1})

clean[["title", "type", "type_encoded", "date_added", "year_added", "duration_value",
       "duration_unit", "primary_country", "primary_genre", "rating_group"]].head()

## Step 5: Create a clean dataset for further analysis

In [ ]:
clean = clean.reset_index(drop=True)

# Final checks
print("Final shape:", clean.shape)
print("Remaining nulls in key columns:", clean[["date_added", "duration_value", "rating_group"]].isna().sum().sum())

# Save (the list column is stored as text for the CSV)
to_save = clean.copy()
to_save["genres"] = to_save["genres"].apply(lambda g: ", ".join(g))
to_save.to_csv("outputs/clean_netflix.csv", index=False)
print("Saved -> outputs/clean_netflix.csv")
clean.head()

---
# Task 2 (Easy): Exploratory Data Analysis
## Step 1: Analyze dataset structure and statistics

In [ ]:
df2 = clean.copy()
print("Rows, columns:", df2.shape)
print("\nUnique values per column:")
print(df2.drop(columns="genres").nunique())
df2[["release_year", "year_added", "duration_value", "num_genres", "years_to_add"]].describe().round(2)

In [ ]:
# Movies (minutes) and TV shows (seasons) must be described separately
print("Movie duration (minutes):")
print(df2[df2["type"] == "Movie"]["duration_value"].describe().round(1))
print("\nTV show seasons:")
print(df2[df2["type"] == "TV Show"]["duration_value"].describe().round(1))

## Step 2: Study content distribution by type

In [ ]:
type_counts = df2["type"].value_counts()
print(type_counts)
print((type_counts / type_counts.sum() * 100).round(1).astype(str) + " %")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df2, x="type", hue="type", palette=["#E50914", "#221F1F"], legend=False, ax=ax[0])
ax[0].set_title("Movies vs TV Shows")
ax[1].pie(type_counts, labels=type_counts.index, autopct="%1.1f%%", colors=["#E50914", "#221F1F"], startangle=90)
ax[1].set_title("Share of content")
plt.tight_layout()
plt.savefig("outputs/eda_1_type_distribution.png", dpi=150)
plt.show()

## Step 3: Identify top countries and categories

In [ ]:
# Count every country a title is associated with (not only the first one)
countries = (df2["country"].str.split(",").explode().str.strip())
top_countries = countries[countries != "Unknown"].value_counts().head(10)

genres = df2["genres"].explode()
top_genres = genres.value_counts().head(10)

directors = df2["director"].str.split(",").explode().str.strip()
top_directors = directors[directors != "Unknown"].value_counts().head(10)

print("Top 10 countries:\n", top_countries, "\n")
print("Top 10 genres:\n", top_genres, "\n")
print("Top 10 directors:\n", top_directors)

## Step 4: Create visualizations for key metrics

In [ ]:
# 4a. Top countries and genres
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, color="#E50914", ax=ax[0])
ax[0].set_title("Top 10 countries by number of titles"); ax[0].set_xlabel("Titles")
sns.barplot(x=top_genres.values, y=top_genres.index, color="#221F1F", ax=ax[1])
ax[1].set_title("Top 10 genres / categories"); ax[1].set_xlabel("Titles")
plt.tight_layout()
plt.savefig("outputs/eda_2_top_countries_genres.png", dpi=150)
plt.show()

In [ ]:
# 4b. Content added to Netflix per year, by type
added = df2.groupby(["year_added", "type"]).size().unstack(fill_value=0)
added.plot(kind="line", marker="o", color=["#E50914", "#221F1F"], figsize=(10, 5))
plt.title("Titles added to Netflix per year"); plt.xlabel("Year added"); plt.ylabel("Titles")
plt.tight_layout()
plt.savefig("outputs/eda_3_titles_added_per_year.png", dpi=150)
plt.show()

In [ ]:
# 4c. Release-year distribution (when was the content made?)
plt.figure(figsize=(10, 5))
sns.histplot(data=df2[df2["release_year"] >= 1990], x="release_year", hue="type", multiple="stack",
             palette=["#E50914", "#221F1F"], bins=32)
plt.title("Release year of titles (1990 onwards)")
plt.tight_layout()
plt.savefig("outputs/eda_4_release_year.png", dpi=150)
plt.show()

In [ ]:
# 4d. Movie length and number of seasons
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df2[df2["type"] == "Movie"]["duration_value"], bins=40, color="#E50914", ax=ax[0])
ax[0].set_title("Movie duration (minutes)"); ax[0].set_xlabel("Minutes")
seasons = df2[df2["type"] == "TV Show"]["duration_value"].value_counts().sort_index()
sns.barplot(x=seasons.index, y=seasons.values, color="#221F1F", ax=ax[1])
ax[1].set_title("TV shows by number of seasons"); ax[1].set_xlabel("Seasons"); ax[1].set_ylabel("Shows")
plt.tight_layout()
plt.savefig("outputs/eda_5_duration.png", dpi=150)
plt.show()

In [ ]:
# 4e. Audience ratings and the month content is added
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
order = ["Kids", "Family", "Teens", "Adults", "Unrated"]
sns.countplot(data=df2, x="rating_group", hue="type", order=order, palette=["#E50914", "#221F1F"], ax=ax[0])
ax[0].set_title("Audience rating group by type")

months = ["January","February","March","April","May","June","July","August","September","October","November","December"]
sns.countplot(data=df2, x="month_name", order=months, color="#E50914", ax=ax[1])
ax[1].set_title("Titles added by month"); ax[1].tick_params(axis="x", rotation=60); ax[1].set_xlabel("")
plt.tight_layout()
plt.savefig("outputs/eda_6_ratings_months.png", dpi=150)
plt.show()

## Step 5: Summarize findings

In [ ]:
movies_pct = (df2["type"] == "Movie").mean() * 100
peak_added = df2["year_added"].value_counts().idxmax()
common_rating = df2["rating"].value_counts().idxmax()
adult_pct = (df2["rating_group"] == "Adults").mean() * 100
median_movie = df2[df2["type"] == "Movie"]["duration_value"].median()
one_season_pct = (df2[df2["type"] == "TV Show"]["duration_value"] == 1).mean() * 100
recent_pct = (df2["release_year"] >= 2015).mean() * 100

print("KEY FINDINGS")
print("-" * 60)
print(f"1. Netflix has {len(df2):,} unique titles: {movies_pct:.1f}% movies and {100 - movies_pct:.1f}% TV shows.")
print(f"2. Top country: {top_countries.index[0]} ({top_countries.iloc[0]:,} titles), then {top_countries.index[1]} and {top_countries.index[2]}.")
print(f"3. Top genres: {top_genres.index[0]}, {top_genres.index[1]} and {top_genres.index[2]}.")
print(f"4. The most titles were added in {peak_added}.")
print(f"5. Most common rating is {common_rating}; {adult_pct:.1f}% of titles are for adults (TV-MA / R / NC-17).")
print(f"6. Median movie length is {median_movie:.0f} minutes; {one_season_pct:.1f}% of TV shows have only 1 season.")
print(f"7. {recent_pct:.1f}% of the catalogue was released in 2015 or later, so the library is mostly recent content.")

---
# Task 3 (Medium): Recommendation System Analysis
A **content-based** recommender: titles that have similar features (genres, country, director, rating, type and title words) are recommended together. We turn the features into numbers with TF-IDF and measure similarity with cosine similarity.

## Step 1: Extract relevant content features

In [ ]:
df3 = clean.reset_index(drop=True).copy()

def clean_token(text):
    # "International Movies" -> "international_movies" so each category stays one token
    return text.strip().lower().replace("&", "and").replace(" ", "_")

def build_features(row):
    tokens = [clean_token(g) for g in row["genres"]]
    tokens.append("type_" + clean_token(row["type"]))
    tokens.append("rating_" + clean_token(row["rating"]))
    tokens += ["country_" + clean_token(c) for c in row["country"].split(",") if c.strip() and c.strip() != "Unknown"]
    if row["director"] != "Unknown":
        tokens += ["dir_" + clean_token(d) for d in row["director"].split(",")]
    return " ".join(tokens)

df3["features"] = df3.apply(build_features, axis=1)
df3[["title", "features"]].head()

## Step 2: Perform text preprocessing

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import hstack

# Titles: lowercase, keep letters only (helps match franchises, e.g. "Motu Patlu ...")
df3["title_clean"] = df3["title"].str.lower().apply(lambda t: re.sub(r"[^a-z\s]", " ", t))

# TF-IDF on the category features (each token is one feature)
tfidf_features = TfidfVectorizer(token_pattern=r"\S+")
X_features = tfidf_features.fit_transform(df3["features"])

# TF-IDF on title words (stop words removed)
tfidf_title = TfidfVectorizer(stop_words="english", token_pattern=r"[a-z]{3,}")
X_title = tfidf_title.fit_transform(df3["title_clean"])

# Combine: category features count more than title words
X = normalize(hstack([X_features, 0.5 * X_title]).tocsr())
print("Feature matrix shape:", X.shape)

## Step 3: Calculate content similarity

In [ ]:
from sklearn.metrics.pairwise import linear_kernel   # rows are normalized, so this equals cosine similarity

# The full 8,700 x 8,700 matrix is large, so similarity is calculated for one title at a time
title_to_idx = pd.Series(df3.index, index=df3["title"].str.lower())

def recommend(title, n=5):
    key = title.lower()
    if key not in title_to_idx.index:
        raise ValueError(f"'{title}' not found in the dataset")
    idx = title_to_idx[key]
    sims = linear_kernel(X[idx], X).ravel()
    sims[idx] = -1                                    # do not recommend the title itself
    top = np.argsort(sims)[::-1][:n]
    out = df3.iloc[top][["title", "type", "listed_in", "country", "rating"]].copy()
    out["similarity"] = sims[top].round(3)
    return out.reset_index(drop=True)

## Step 4: Generate recommendations for selected titles

In [ ]:
for t in ["Ganglands", "Midnight Mass", "The Great British Baking Show", "Motu Patlu in Wonderland"]:
    print(f"\n=== Because you watched: {t} ===")
    display(recommend(t, n=5))

## Step 5: Evaluate recommendation quality

In [ ]:
# We check 1,000 random titles. For each one, take the top 5 recommendations and measure:
#   - same type rate        : recommended title is the same type (Movie / TV Show)
#   - genre match rate      : shares at least one genre with the original title
#   - average genre Jaccard : overlap of the two genre sets (1.0 = identical genres)
# A random recommender is used as the baseline for comparison.
rng = np.random.default_rng(42)
sample = rng.choice(len(df3), size=1000, replace=False)

sim_matrix = linear_kernel(X[sample], X)
sim_matrix[np.arange(len(sample)), sample] = -1
top5 = np.argsort(-sim_matrix, axis=1)[:, :5]
random5 = rng.integers(0, len(df3), size=top5.shape)

genre_sets = df3["genres"].apply(set).values
types = df3["type"].values

def score(rec_idx):
    same_type, match, jacc = [], [], []
    for q, recs in zip(sample, rec_idx):
        for r in recs:
            same_type.append(types[q] == types[r])
            inter = len(genre_sets[q] & genre_sets[r]); union = len(genre_sets[q] | genre_sets[r])
            match.append(inter > 0); jacc.append(inter / union)
    return np.mean(same_type), np.mean(match), np.mean(jacc)

results = pd.DataFrame(
    {"Content-based model": score(top5), "Random baseline": score(random5)},
    index=["Same type rate", "Genre match rate", "Avg genre Jaccard"]).round(3)
results

In [ ]:
ax = results.T.plot(kind="bar", figsize=(9, 4.5), color=["#E50914", "#221F1F", "#B3B3B3"], rot=0)
ax.set_title("Recommendation quality vs random baseline"); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("outputs/rec_quality.png", dpi=150)
plt.show()

print("Note: genres are part of the features, so a high genre score is expected. It confirms the model")
print("recommends similar content; real user feedback (clicks / watch time) would be needed to judge taste.")

---
# Task 4 (Medium): Trend Prediction Analysis
## Step 1: Prepare release-year data

In [ ]:
# 2021 is only a partial year (the data ends in September 2021), and years before 2000 have very few titles.
# So we model 2000 - 2020.
yearly = clean.groupby("release_year").size().loc[2000:2020].rename("titles")
yearly_type = clean.groupby(["release_year", "type"]).size().unstack(fill_value=0).loc[2000:2020]
print(yearly.tail(8))

## Step 2: Analyze yearly content trends

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(yearly.index, yearly.values, marker="o", color="#E50914")
ax[0].set_title("Titles released per year (on Netflix)"); ax[0].set_xlabel("Release year")
yearly_type.plot(kind="area", stacked=True, color=["#E50914", "#221F1F"], alpha=0.85, ax=ax[1])
ax[1].set_title("Movies vs TV shows released per year"); ax[1].set_xlabel("Release year")
plt.tight_layout()
plt.savefig("outputs/trend_1_history.png", dpi=150)
plt.show()

growth = (yearly.pct_change() * 100).round(1)
peak_year = yearly.idxmax()
print(f"Peak year: {peak_year} with {yearly.max()} titles")
print("Year-over-year growth (%), last 8 years:")
print(growth.tail(8))

## Step 3: Build forecasting models
We compare five simple models and a naive baseline ("next years equal the last known year"). Models are trained on **2000–2017** and tested on the last three years **2018–2020**, so we can see how well each one predicts data it has not seen.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

years = yearly.index.values
y = yearly.values.astype(float)
X_year = (years - 2000).reshape(-1, 1)          # years since 2000 keeps the numbers small

train = years <= 2017
test = ~train

# ---- Model 1 & 2: linear and polynomial (degree 2) trend
def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), LinearRegression())

# ---- Model 3: exponential growth (linear regression on log of the counts)
class ExpTrend:
    def fit(self, X, y):
        self.m = LinearRegression().fit(X, np.log(y)); return self
    def predict(self, X):
        return np.exp(self.m.predict(X))

# ---- Model 4 & 5: Holt's exponential smoothing (level + trend). phi < 1 "damps" the trend so growth fades out.
def holt_forecast(series, steps, alpha, beta, phi=1.0):
    level, trend = series[0], series[1] - series[0]
    for t in range(1, len(series)):
        prev = level
        level = alpha * series[t] + (1 - alpha) * (level + phi * trend)
        trend = beta * (level - prev) + (1 - beta) * phi * trend
    return np.array([level + sum(phi ** k for k in range(1, i + 2)) * trend for i in range(steps)])

def holt_one_step_error(series, alpha, beta, phi=1.0):
    level, trend, err = series[0], series[1] - series[0], 0.0
    for t in range(1, len(series)):
        err += (series[t] - (level + phi * trend)) ** 2
        prev = level
        level = alpha * series[t] + (1 - alpha) * (level + phi * trend)
        trend = beta * (level - prev) + (1 - beta) * phi * trend
    return err

# choose the smoothing parameters using the training data only
grid = np.round(np.arange(0.1, 1.0, 0.1), 1)
phis = [0.7, 0.8, 0.9, 0.95]
best_alpha, best_beta = min(((a, b) for a in grid for b in grid), key=lambda ab: holt_one_step_error(y[train], *ab))
d_alpha, d_beta, d_phi = min(((a, b, p) for a in grid for b in grid for p in phis),
                             key=lambda abp: holt_one_step_error(y[train], *abp))
print(f"Holt: alpha={best_alpha}, beta={best_beta}")
print(f"Damped Holt: alpha={d_alpha}, beta={d_beta}, phi={d_phi}")

# ---- Predict the test years (2018-2020) with every model
test_pred = {}
for name, model in {"Linear trend": poly_model(1), "Polynomial (deg 2)": poly_model(2), "Exponential growth": ExpTrend()}.items():
    model.fit(X_year[train], y[train])
    test_pred[name] = model.predict(X_year[test])
test_pred["Holt smoothing"] = holt_forecast(y[train], test.sum(), best_alpha, best_beta)
test_pred["Damped Holt"] = holt_forecast(y[train], test.sum(), d_alpha, d_beta, d_phi)
test_pred["Naive (last value)"] = np.repeat(y[train][-1], test.sum())   # simple baseline: next years = last known year

### Model evaluation on the unseen years (2018–2020)

In [ ]:
rows = []
for name, pred in test_pred.items():
    rows.append({"Model": name,
                 "MAE": mean_absolute_error(y[test], pred),
                 "RMSE": np.sqrt(mean_squared_error(y[test], pred)),
                 "MAPE %": np.mean(np.abs((y[test] - pred) / y[test])) * 100})
scores = pd.DataFrame(rows).set_index("Model").round(1).sort_values("RMSE")
display(scores)

best_model = scores.index[0]
print("Best model on the test years:", best_model)

actual_vs_pred = pd.DataFrame(test_pred, index=years[test]).round(0)
actual_vs_pred.insert(0, "Actual", y[test])
actual_vs_pred

## Step 4: Visualize future predictions
Now every model is re-trained on all years (2000–2020) and used to forecast **2021–2025**.

In [ ]:
future_years = np.arange(2021, 2026)
X_future = (future_years - 2000).reshape(-1, 1)

# Re-fit Holt on all data (alpha, beta chosen on all data)
a_all, b_all = min(((a, b) for a in grid for b in grid), key=lambda ab: holt_one_step_error(y, *ab))
da_all, db_all, dp_all = min(((a, b, p) for a in grid for b in grid for p in phis),
                             key=lambda abp: holt_one_step_error(y, *abp))

forecasts = {}
for name, model in {"Linear trend": poly_model(1), "Polynomial (deg 2)": poly_model(2), "Exponential growth": ExpTrend()}.items():
    forecasts[name] = model.fit(X_year, y).predict(X_future)
forecasts["Holt smoothing"] = holt_forecast(y, len(future_years), a_all, b_all)
forecasts["Damped Holt"] = holt_forecast(y, len(future_years), da_all, db_all, dp_all)
forecasts["Naive (last value)"] = np.repeat(y[-1], len(future_years))

forecast_table = pd.DataFrame(forecasts, index=future_years).round(0).astype(int)
forecast_table.index.name = "Year"
display(forecast_table)

plt.figure(figsize=(11, 5.5))
plt.plot(years, y, "o-", color="black", label="Actual")
colors = {"Linear trend": "#1f77b4", "Polynomial (deg 2)": "#2ca02c", "Exponential growth": "#ff7f0e", "Holt smoothing": "#E50914",
          "Damped Holt": "#9467bd", "Naive (last value)": "#7f7f7f"}
for name, pred in forecasts.items():
    lw = 3 if name == best_model else 1.5
    plt.plot(np.r_[years[-1], future_years], np.r_[y[-1], pred], "--", color=colors[name], linewidth=lw,
             label=f"{name}" + (" (best on test)" if name == best_model else ""))
plt.axvline(2020.5, color="grey", linestyle=":")
plt.title("Titles released per year: history and forecast to 2025")
plt.xlabel("Release year"); plt.ylabel("Titles")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/trend_2_forecast.png", dpi=150)
plt.show()

## Step 5: Interpret results

In [ ]:
last3 = yearly.loc[2018:2020]
peak = yearly.max()
unrealistic = [n for n in forecast_table.columns if forecast_table.loc[2025, n] > 1.5 * peak]
sensible = scores.drop(index=unrealistic, errors="ignore").index[0]

print("INTERPRETATION")
print("-" * 60)
print(f"1. Content released per year grew from {yearly.loc[2000]} titles in 2000 to a peak of {peak} in {peak_year}.")
print(f"2. After {peak_year} the numbers fell: {last3.loc[2018]} (2018) -> {last3.loc[2019]} (2019) -> {last3.loc[2020]} (2020).")
print(f"3. Lowest error on the unseen years 2018-2020: '{best_model}' "
      f"(RMSE {scores.loc[best_model, 'RMSE']}, MAPE {scores.loc[best_model, 'MAPE %']}%).")
if unrealistic:
    print(f"4. But {', '.join(unrealistic)} forecast more than 1.5x the historical peak ({peak}) by 2025, which is not realistic.")
    print(f"   Best model with a realistic forecast: '{sensible}' -> {forecast_table.loc[2025, sensible]} titles in 2025.")
else:
    print(f"4. Best model: '{sensible}' -> {forecast_table.loc[2025, sensible]} titles forecast for 2025.")
print(f"5. Forecasts for 2025 range from {forecast_table.loc[2025].min()} to {forecast_table.loc[2025].max()} titles, so the model choice matters a lot.")
print("6. Caution: the series turned downward after the peak (possible reasons: the 2020 pandemic, and the catalogue")
print("   only containing titles still on Netflix). With about 20 yearly points, treat forecasts as a rough guide.")